# Figuras del capítulo 4 — Curación del conjunto (OE1)

Generado al dividir `notebooks/figuras_documento.ipynb` por capítulo.
Cada celda es independiente después de la sección 0: se puede correr solo la que cambió.


## 0. Preparación: rutas, estilo y helpers

Las figuras se escriben en `../figures/` de este mismo capítulo.


In [ ]:
import json
import logging
import sys
import textwrap
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch, PathPatch, Rectangle
from matplotlib.path import Path as MplPath

NB_DIR = Path.cwd()
CHAPTER_DIR = NB_DIR.parent
PROJECT_DIR = next(d for d in (NB_DIR, *NB_DIR.parents) if (d / "pyproject.toml").exists())
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from core.config import CLEANED_DIR, PROCESSED_DIR, RAW_DIR, RUNS_DIR, P  # noqa: E402

logging.getLogger("data.annotations").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

FIGURE_DIR = CHAPTER_DIR / "figures"  # capitulos/<capitulo>/figures
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = PROCESSED_DIR
COMPARISON_DIR = RUNS_DIR / "comparacion"

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
# --- Estilo ------------------------------------------------------------------
SURFACE = "#ffffff"
INK = "#0b0b0b"
INK_2 = "#525252"
MUTED = "#8a8a8a"
GRID = "#e3e3e3"
BASELINE = "#c4c4c4"

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # slots categóricos 1-4
CRITICAL = "#d03b3b"
WARNING = "#eda100"
GOOD = "#1baf7a"
BLUE = "#2a78d6"
BLUE_FADED = "#b7d3f6"
BLUE_INK = "#0d366b"
PANEL = "#f3f3f3"  # relleno de caja de diagrama

mpl.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "savefig.edgecolor": SURFACE,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 9,
        "text.color": INK,
        "axes.edgecolor": BASELINE,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 10,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "lines.linewidth": 2,
    }
)


def style_value_axis(ax, axis="x"):
    """Rejilla fina sólo en el eje de valor; sin espina de línea base en ese eje."""
    ax.grid(True, axis=axis, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    if axis == "x":
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_color(BASELINE)
    else:
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_color(BASELINE)


_K = 0.5523  # constante de control del arco circular en una Bezier cúbica


def _square_path(x0, y0, w, h):
    return MplPath(
        [(x0, y0), (x0 + w, y0), (x0 + w, y0 + h), (x0, y0 + h), (x0, y0)],
        [MplPath.MOVETO] + [MplPath.LINETO] * 3 + [MplPath.CLOSEPOLY],
    )


def _rounded_path(x0, y0, w, h, rx, ry, side="right"):
    """Rectángulo con las dos esquinas de `side` redondeadas: el extremo de dato."""
    rx, ry = min(abs(rx), abs(w)), min(abs(ry), abs(h) / 2)
    if rx <= 0 or ry <= 0 or w == 0 or h == 0:
        return _square_path(x0, y0, w, h)
    x1, y1 = x0 + w, y0 + h

    if side == "right":
        verts = [
            (x0, y0),
            (x1 - rx, y0),
            (x1 - rx + rx * _K, y0),
            (x1, y0 + ry - ry * _K),
            (x1, y0 + ry),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0, y1),
            (x0, y0),
        ]
    else:  # "top"
        verts = [
            (x0, y0),
            (x1, y0),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0 + rx, y1),
            (x0 + rx - rx * _K, y1),
            (x0, y1 - ry + ry * _K),
            (x0, y1 - ry),
            (x0, y0),
        ]
    codes = [
        MplPath.MOVETO,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CLOSEPOLY,
    ]
    return MplPath(verts, codes)


def _x_per_y(ax):
    """Unidades de x que se dibujan tan largas como una unidad de y."""
    box = ax.get_position()
    width_in = ax.figure.get_figwidth() * box.width
    height_in = ax.figure.get_figheight() * box.height
    x_range = abs(np.diff(ax.get_xlim())[0])
    y_range = abs(np.diff(ax.get_ylim())[0])
    return (x_range / width_in) / (y_range / height_in)


def barh(ax, ys, widths, color, height=0.62, left=None, radius_frac=0.30, **kw):
    """Barras horizontales con el extremo de dato redondeado. `left` apila."""
    ys = np.atleast_1d(np.asarray(ys, dtype=float))
    widths = np.asarray(widths, dtype=float)
    left = np.zeros_like(widths) if left is None else np.asarray(left, dtype=float)
    ry = height * radius_frac
    rx = ry * _x_per_y(ax)
    for y, w, x0 in zip(ys, widths, left):
        ax.add_patch(
            PathPatch(
                _rounded_path(x0, y - height / 2, w, height, rx, ry, "right"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


def barv(ax, xs, heights, color, width=0.68, radius_frac=0.30, bottom=None, **kw):
    """Barras verticales con la punta redondeada."""
    xs = np.atleast_1d(np.asarray(xs, dtype=float))
    heights = np.asarray(heights, dtype=float)
    bottom = np.zeros_like(heights) if bottom is None else np.asarray(bottom, dtype=float)
    rx = width * radius_frac
    ry = rx / _x_per_y(ax)
    for x, h, y0 in zip(xs, heights, bottom):
        ax.add_patch(
            PathPatch(
                _rounded_path(x - width / 2, y0, width, h, rx, ry, "top"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


# --- Salida ------------------------------------------------------------------
# Dos artefactos por figura: el PNG que `main.tex` incluye hoy y, cuando la figura es
# una serie de datos y no un diagrama, el JSON con esos datos, para poder redibujarla
# en pgfplots sin volver a correr el cuaderno.
DATA_DIR = FIGURE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Series, pd.Index)):
        return value.tolist()
    if isinstance(value, pd.DataFrame):
        return value.to_dict(orient="list")
    raise TypeError(type(value))


def save(fig, name, data=None):
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path)
    if data is not None:
        target = DATA_DIR / f"{name}.json"
        target.write_text(json.dumps(data, indent=1, ensure_ascii=False, default=_jsonable))

In [ ]:
# --- Helpers de diagrama ------------------------------------------------------


def canvas(width_in=10.0, height_in=5.5):
    fig, ax = plt.subplots(figsize=(width_in, height_in))
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100 * height_in / width_in)
    ax.set_aspect("equal")
    ax.axis("off")
    return fig, ax


def wrap_text(text, width=None):
    """Ajusta a `width` respetando los saltos de línea que ya trae el texto."""
    text = str(text)
    if not width:
        return text
    return "\n".join(
        textwrap.fill(line, width) if line.strip() else line for line in text.split("\n")
    )


def _units_per_point(ax):
    """Unidades del lienzo que mide un punto tipográfico (lienzo de 100 de ancho)."""
    return 100.0 / (ax.figure.get_figwidth() * 72.0)


def _stack(ax, blocks):
    """Alturas y separaciones de una pila de bloques de texto, en unidades del lienzo."""
    upt = _units_per_point(ax)
    heights = [(text.count("\n") + 1) * size * 1.34 * upt for text, size, _, _ in blocks]
    gaps = [1.15 * (blocks[i][1] + blocks[i + 1][1]) / 2 * upt for i in range(len(blocks) - 1)]
    return heights, gaps, sum(heights) + sum(gaps)


def _blocks(
    title, body, foot, fs, body_fs, foot_fs, ink, body_ink, foot_ink, weight, wrap, body_wrap
):
    blocks = []
    if title:
        blocks.append((wrap_text(title, wrap), fs, ink, weight))
    if body:
        blocks.append((wrap_text(body, body_wrap or wrap or 30), body_fs, body_ink, "normal"))
    if foot:
        blocks.append((wrap_text(foot, body_wrap or wrap or 30), foot_fs, foot_ink, "normal"))
    return blocks


def node(
    ax,
    cx,
    cy,
    w,
    h=None,
    title="",
    body=None,
    foot=None,
    *,
    fill=PANEL,
    edge=BASELINE,
    ink=INK,
    body_ink=INK_2,
    foot_ink=MUTED,
    lw=1.1,
    fs=9.0,
    body_fs=7.8,
    foot_fs=7.0,
    weight="bold",
    dashed=False,
    wrap=None,
    body_wrap=None,
    radius=1.4,
    pad_pt=9.0,
    zorder=2,
):
    """Caja redondeada centrada en (cx, cy) con hasta tres bloques de texto apilados.

    Con `h=None` la caja se ajusta al texto: es lo que evita las cajas medio vacías y
    los títulos encimados cuando el contenido cambia de largo.
    """
    blocks = _blocks(
        title, body, foot, fs, body_fs, foot_fs, ink, body_ink, foot_ink, weight, wrap, body_wrap
    )
    heights, gaps, total = _stack(ax, blocks)
    if h is None:
        h = total + 2 * pad_pt * _units_per_point(ax)

    ax.add_patch(
        FancyBboxPatch(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            boxstyle=f"round,pad=0,rounding_size={radius}",
            linewidth=lw,
            edgecolor=edge,
            facecolor=fill,
            linestyle=(0, (4, 3)) if dashed else "-",
            zorder=zorder,
            mutation_aspect=1,
        )
    )

    y = cy + total / 2
    for i, (text, size, color, wt) in enumerate(blocks):
        ax.text(
            cx,
            y - heights[i] / 2,
            text,
            ha="center",
            va="center",
            fontsize=size,
            color=color,
            fontweight=wt,
            zorder=zorder + 1,
            linespacing=1.32,
        )
        y -= heights[i] + (gaps[i] if i < len(gaps) else 0)
    return cx, cy, w, h


def node_height(
    ax,
    title,
    body=None,
    foot=None,
    *,
    fs=9.0,
    body_fs=7.8,
    foot_fs=7.0,
    wrap=None,
    body_wrap=None,
    pad_pt=9.0,
):
    blocks = _blocks(
        title, body, foot, fs, body_fs, foot_fs, INK, INK_2, MUTED, "bold", wrap, body_wrap
    )
    return _stack(ax, blocks)[2] + 2 * pad_pt * _units_per_point(ax)


def flow_row(ax, items, cy, *, margin=2.5, gap=2.6, h=None, arrow_color=BLUE, arrows=True, **kw):
    """Fila de cajas que reparte el ancho del lienzo, con flechas entre ellas.

    `items` es una lista de dicts con `title` y, opcionalmente, `body`, `foot` y `kw`.
    El alto es común a toda la fila: el de la caja que más texto lleva.
    """
    n = len(items)
    w = (100 - 2 * margin - (n - 1) * gap) / n
    if h is None:
        h = max(
            node_height(
                ax,
                item["title"],
                item.get("body"),
                item.get("foot"),
                fs=kw.get("fs", 9.0),
                body_fs=kw.get("body_fs", 7.8),
                foot_fs=kw.get("foot_fs", 7.0),
                wrap=item.get("kw", {}).get("wrap", kw.get("wrap")),
                body_wrap=kw.get("body_wrap"),
            )
            for item in items
        )
    boxes = []
    for i, item in enumerate(items):
        cx = margin + w / 2 + i * (w + gap)
        options = dict(kw)
        options.update(item.get("kw", {}))
        boxes.append(
            node(ax, cx, cy, w, h, item["title"], item.get("body"), item.get("foot"), **options)
        )
        if arrows and i:
            arrow(ax, (cx - w / 2 - gap, cy), (cx - w / 2, cy), color=arrow_color, lw=1.3)
    return boxes


def arrow(ax, start, end, *, color=MUTED, lw=1.2, rad=0.0, dashed=False, head="-|>", zorder=1):
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        zorder=zorder,
        arrowprops=dict(
            arrowstyle=head,
            color=color,
            linewidth=lw,
            shrinkA=1.5,
            shrinkB=1.5,
            linestyle=(0, (4, 3)) if dashed else "-",
            connectionstyle=f"arc3,rad={rad}",
            mutation_scale=11,
        ),
    )


def elbow(ax, points, *, color=BLUE, lw=1.3):
    """Conector en ángulo recto con punta de flecha en el último tramo."""
    xs, ys = zip(*points)
    ax.plot(xs, ys, color=color, linewidth=lw, solid_capstyle="round", zorder=1)
    arrow(ax, points[-2], points[-1], color=color, lw=lw)


def caption_at(ax, x, y, text, *, fs=7.8, color=INK_2, ha="center", va="center", wrap=None, **kw):
    ax.text(
        x,
        y,
        wrap_text(text, wrap),
        ha=ha,
        va=va,
        fontsize=fs,
        color=color,
        linespacing=1.35,
        **kw,
    )


def chip(ax, cx, cy, text, color=BLUE, *, fs=7.5, pad=1.0, ink=SURFACE):
    """Etiqueta pequeña de fondo lleno: para marcar una salida o un estado."""
    ax.text(
        cx,
        cy,
        text,
        ha="center",
        va="center",
        fontsize=fs,
        color=ink,
        fontweight="bold",
        zorder=5,
        bbox=dict(boxstyle=f"round,pad={pad * 0.3}", facecolor=color, edgecolor="none"),
    )


# --- Glifos de tensor ---------------------------------------------------------
# `node` dibuja una caja con texto: sirve para un módulo, no para el dato que viaja
# entre módulos. Esto es el vocabulario que faltaba -- el espectrograma como plano, los
# tokens contables, la pirámide a escala -- para que un diagrama muestre la forma del
# tensor en vez de describirla por escrito.


def slab(ax, cx, cy, w, h, *, skew=0.16, fill=SURFACE, edge=INK, lw=1.2, zorder=2):
    """Plano en perspectiva: el espectrograma y los mapas de salida."""
    dx = w * skew
    points = [
        (cx - w / 2 + dx, cy - h / 2),
        (cx + w / 2, cy - h / 2),
        (cx + w / 2 - dx, cy + h / 2),
        (cx - w / 2, cy + h / 2),
    ]
    ax.add_patch(
        plt.Polygon(points, closed=True, facecolor=fill, edgecolor=edge, lw=lw, zorder=zorder)
    )
    return points


def token_strip(
    ax, cx, cy, w, h, n, *, vertical=True, fill=SURFACE, edge=INK, lw=0.9, gap=0.30, zorder=3
):
    """Tokens dibujados de a uno: se ven y se cuentan, no se leen en un pie de caja."""
    if vertical:
        cell = h / n
        for k in range(n):
            ax.add_patch(
                Rectangle(
                    (cx - w / 2, cy - h / 2 + k * cell + gap / 2),
                    w,
                    cell - gap,
                    facecolor=fill,
                    edgecolor=edge,
                    lw=lw,
                    zorder=zorder,
                )
            )
    else:
        cell = w / n
        for k in range(n):
            ax.add_patch(
                Rectangle(
                    (cx - w / 2 + k * cell + gap / 2, cy - h / 2),
                    cell - gap,
                    h,
                    facecolor=fill,
                    edgecolor=edge,
                    lw=lw,
                    zorder=zorder,
                )
            )


def feature_pyramid(
    ax, cx, cy, w, h, levels, *, fill=SURFACE, edge=INK, lw=1.1, label_y=None, zorder=3
):
    """Un plano por nivel, con el alto proporcional a su resolución.

    La proporción entre los niveles *es* el dato: dibujarlos todos iguales borra lo
    único que distingue una pirámide multiescala de tres convoluciones en fila.
    """
    tallest = max(rows for rows, _ in levels)
    slot = w / len(levels)
    for k, (rows, cols) in enumerate(levels):
        x = cx - w / 2 + slot * (k + 0.5)
        height = h * (rows / tallest) ** 0.55
        ax.add_patch(
            Rectangle(
                (x - slot * 0.22, cy - height / 2),
                slot * 0.44,
                height,
                facecolor=fill,
                edgecolor=edge,
                lw=lw,
                zorder=zorder,
            )
        )
        y = cy - h / 2 - 1.4 if label_y is None else label_y
        caption_at(ax, x, y, f"{rows}x{cols}", fs=6.0, color=MUTED, va="top")


def patch_grid(ax, cx, cy, w, h, n_cols, n_rows, *, edge=INK, lw=0.9, zorder=3):
    """La rejilla de parches sobre la entrada: el parcheo del AST hecho visible."""
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor=SURFACE,
            edgecolor=edge,
            lw=lw,
            zorder=zorder,
        )
    )
    for i in range(1, n_cols):
        x = cx - w / 2 + w * i / n_cols
        ax.plot([x, x], [cy - h / 2, cy + h / 2], color=edge, lw=lw * 0.7, zorder=zorder + 1)
    for j in range(1, n_rows):
        y = cy - h / 2 + h * j / n_rows
        ax.plot([cx - w / 2, cx + w / 2], [y, y], color=edge, lw=lw * 0.7, zorder=zorder + 1)


def detection_slab(ax, cx, cy, w, h, boxes, *, color=CRITICAL, **kw):
    """El plano de salida con las cajas encima: dónde termina el pipeline."""
    slab(ax, cx, cy, w, h, **kw)
    for bx, by, bw, bh in boxes:
        ax.add_patch(
            Rectangle(
                (cx - w / 2 + (bx - bw / 2) * w, cy - h / 2 + (by - bh / 2) * h),
                bw * w,
                bh * h,
                facecolor="none",
                edgecolor=color,
                lw=1.3,
                zorder=6,
            )
        )


def zoom_arrow(ax, cx, cy, w, h, *, fill=PANEL, edge=MUTED, lw=1.0, zorder=1):
    """Flecha hueca hacia abajo: 'esto de arriba se abre en este panel'."""
    shaft, head = w * 0.42, h * 0.46
    points = [
        (cx - shaft / 2, cy + h / 2),
        (cx + shaft / 2, cy + h / 2),
        (cx + shaft / 2, cy - h / 2 + head),
        (cx + w / 2, cy - h / 2 + head),
        (cx, cy - h / 2),
        (cx - w / 2, cy - h / 2 + head),
        (cx - shaft / 2, cy - h / 2 + head),
    ]
    ax.add_patch(
        plt.Polygon(points, closed=True, facecolor=fill, edgecolor=edge, lw=lw, zorder=zorder)
    )


def detail_panel(ax, cx, cy, w, h, title="", *, edge=BASELINE, lw=1.1, fs=7.6, zorder=0):
    """El recuadro que enmarca un panel de detalle del segundo piso."""
    ax.add_patch(
        FancyBboxPatch(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            boxstyle="round,pad=0,rounding_size=1.6",
            facecolor=SURFACE,
            edgecolor=edge,
            lw=lw,
            zorder=zorder,
        )
    )
    if title:
        caption_at(
            ax,
            cx - w / 2 + 2.0,
            cy + h / 2 - 2.2,
            title,
            fs=fs,
            color=INK,
            ha="left",
            va="top",
            weight="bold",
        )

In [ ]:
import soundfile as sf  # noqa: E402
import torch  # noqa: E402
import torchaudio  # noqa: E402

from utils.audio import hz_to_y, mel_spectrogram, y_to_hz  # noqa: E402

MEL = mel_spectrogram(P)


def load_segment(audio_path, start_s=0.0, duration_s=3.0, params=P):
    """Trozo mono de un `.wav`, remuestreado a `params.target_sr`."""
    with sf.SoundFile(str(audio_path)) as handle:
        handle.seek(int(start_s * handle.samplerate))
        frames = handle.read(int(duration_s * handle.samplerate), dtype="float32", always_2d=True)
        waveform = torch.from_numpy(frames.mean(axis=1))
        if handle.samplerate != params.target_sr:
            waveform = torchaudio.functional.resample(waveform, handle.samplerate, params.target_sr)
    return waveform


def log_mel(waveform, params=P):
    return np.log(MEL(waveform).numpy() + params.eps)


def draw_spectrogram(ax, image, duration_s, *, params=P, cmap="magma", n_ticks=5, vrange=(2, 99.5)):
    """Espectrograma log-mel con el eje de frecuencia en el espacio mel normalizado."""
    lo, hi = np.percentile(image, vrange)
    ax.imshow(
        image,
        origin="lower",
        aspect="auto",
        cmap=cmap,
        extent=(0, duration_s, 0, 1),
        interpolation="nearest",
        vmin=lo,
        vmax=hi,
    )
    ticks = np.linspace(0, 1, n_ticks)
    ax.set_yticks(ticks, [f"{y_to_hz(np.array([t]), params)[0] / 1000:.1f}" for t in ticks])
    ax.set_ylabel("frecuencia (kHz, eje mel)")
    ax.set_xlabel("tiempo (s)")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return ax


def draw_box(
    ax,
    t0,
    t1,
    low_hz,
    high_hz,
    *,
    params=P,
    color=GOOD,
    label=None,
    lw=1.5,
    dashed=False,
    fs=7.5,
    label_below=False,
):
    """Caja tiempo–frecuencia sobre un espectrograma dibujado por `draw_spectrogram`."""
    y0 = float(hz_to_y(np.array([max(low_hz, params.f_min)]), params)[0])
    y1 = float(hz_to_y(np.array([min(high_hz, params.f_max)]), params)[0])
    ax.add_patch(
        FancyBboxPatch(
            (t0, y0),
            t1 - t0,
            y1 - y0,
            boxstyle="round,pad=0,rounding_size=0.012",
            linewidth=lw,
            edgecolor=color,
            facecolor="none",
            linestyle=(0, (4, 2)) if dashed else "-",
            zorder=4,
        )
    )
    if label:
        ax.text(
            t0,
            (y0 - 0.022) if label_below else (y1 + 0.014),
            label,
            color=color,
            fontsize=fs,
            fontweight="bold",
            va="top" if label_below else "bottom",
            ha="left",
            zorder=5,
        )
    return y0, y1

In [ ]:
from data.annotations import load_annotations  # noqa: E402
from data.species import (  # noqa: E402
    CALL_TYPES,
    LABEL_SEPARATOR,
    VALID_PAIRS,  # noqa: E402
    Species,
)
from prepare_data import EXCLUDED_LABELS, MIN_PAIR_COUNT, select_experiment  # noqa: E402

annotations = load_annotations()
annotations["pair"] = annotations.species + LABEL_SEPARATOR + annotations.call_type

SPECIES_NAME = {s.name.lower(): s.value.replace("_", " ") for s in Species}
SPECIES_LATIN = {
    "aa": "Aotus azarae",
    "ac": "Ateles chamek",
    "as": "Alouatta sara",
    "cc": "Cebus cuscinus",
    "lw": "Leontocebus weddelli",
    "pt": "Plecturocebus toppini",
    "sb": "Saimiri boliviensis",
    "sm": "Sapajus macrocephalus",
}
CALL_NAME = {
    f"{s.name.lower()}/{code}": label.replace("_", " ")
    for s, codes in CALL_TYPES.items()
    for code, label in codes.items()
}
CALL_NAME["lw/trino"] = "trino (tr, tj, tt y tf fusionados)"


# El mismo filtro que materializa el caché (`prepare_data.select_experiment`).
experiment, LABELS = select_experiment(annotations)
experiment = experiment.copy()
experiment["pair"] = experiment["label"]
CLASSES = LABELS.names


def recording_table():
    """Duración de cada .wav de raw/ y si tiene anotación en cleaned/ (leerlas tarda minutos)."""
    cache = DATA_DIR / "recording_durations.csv"
    if cache.exists():
        return pd.read_csv(cache)
    rows = []
    for wav in sorted(RAW_DIR.rglob("*.wav")):
        try:
            info = sf.info(str(wav))
        except Exception:
            continue
        folder = next((p.name for p in wav.parents if "__" in p.name), wav.parent.name)
        cleaned = (CLEANED_DIR / wav.relative_to(RAW_DIR)).with_suffix(".txt")
        rows.append(
            {
                "audio_path": str(wav),
                "species": folder.split("__")[-1].lower(),
                "duration_s": info.duration,
                "has_annotation": cleaned.exists(),
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(cache, index=False)
    return table


recordings = recording_table()

In [ ]:
# --- Elegir ejemplos reales de forma determinista -----------------------------


def pick_annotation(pair, *, df=None, rank=0, source=None):
    """Anotación de `pair` cuya duración es la `rank`-ésima más cercana a la mediana."""
    table = annotations if df is None else df
    rows = table[table.pair == pair]
    if source is not None:
        rows = rows[rows.audio_path.str.contains(source)]
    if rows.empty:
        raise LookupError(f"sin anotaciones para {pair}")
    order = (rows.duration_s - rows.duration_s.median()).abs().sort_values().index
    return rows.loc[order[min(rank, len(order) - 1)]]


def busiest_segment(pair, window_s=6.0, *, df=None, n_events=6):
    """Ventana de `window_s` con un número de eventos de `pair` cercano a `n_events`."""
    table = annotations if df is None else df
    rows = table[table.pair == pair]
    best = []
    for audio_path, group in rows.groupby("audio_path"):
        for start in group.begin_time_s.to_numpy():
            inside = group[(group.begin_time_s >= start) & (group.end_time_s <= start + window_s)]
            if len(inside):
                best.append(
                    (
                        abs(len(inside) - n_events),
                        -float(inside.duration_s.sum()),
                        audio_path,
                        start,
                    )
                )
    best.sort()
    _, _, audio_path, start = best[0]
    inside = table[
        (table.audio_path == audio_path)
        & (table.end_time_s > start)
        & (table.begin_time_s < start + window_s)
    ]
    return audio_path, float(start), inside


def segment_around(row, pad_ratio=0.9, min_pad_s=0.25, params=P):
    """Inicio y duración de un trozo centrado en la anotación `row`."""
    pad = max(row.duration_s * pad_ratio, min_pad_s)
    start = max(row.begin_time_s - pad, 0.0)
    return start, float(row.duration_s + 2 * pad)

## 4. Curación del conjunto de datos (OE1)


In [ ]:
# Anotaciones por par especie/llamada en cleaned/, y cuáles superan el umbral del experimento.
counts = annotations.pair.value_counts()
selected = counts.index.isin(CLASSES)
order = counts.index.tolist()
ys = np.arange(len(order))

fig, ax = plt.subplots(figsize=(9.0, 0.24 * len(order) + 1.2))
ax.set_xscale("log")
ax.set_xlim(1, counts.max() * 1.6)
ax.set_ylim(len(order) - 0.4, -0.8)
for y, value, chosen in zip(ys, counts.values, selected):
    ax.plot([1, value], [y, y], color=GRID, linewidth=1.0, zorder=1)
    ax.scatter([value], [y], s=22, color=BLUE if chosen else MUTED, zorder=3)
    ax.text(value * 1.12, y, f"{value:,}", va="center", fontsize=6.8, color=INK_2 if chosen else MUTED)
ax.axvline(MIN_PAIR_COUNT, color=SERIES[1], linewidth=1.2, linestyle=(0, (4, 3)), zorder=2)
ax.text(MIN_PAIR_COUNT * 1.05, -0.5, f"umbral {MIN_PAIR_COUNT}", color=SERIES[1], fontsize=8, va="center")
ax.set_yticks(ys, order, fontsize=7.2)
for label, chosen in zip(ax.get_yticklabels(), selected):
    label.set_color(INK if chosen else MUTED)
style_value_axis(ax, "x")
ax.set_xlabel("anotaciones (escala log)")

save(
    fig,
    "annotations_per_pair",
    data={"par": order, "anotaciones": counts.values, "seleccionada": selected, "umbral": MIN_PAIR_COUNT},
)
plt.show()

In [ ]:
# Geometría por clase: duración (cuartiles, escala log) y banda de frecuencia (mediana de
# low/high y su rango intercuartil). La Tabla de `main.tex` toma las mismas cifras del JSON.
by_class = experiment.groupby("pair")
geometry = pd.DataFrame(
    {
        "q1": by_class.duration_s.quantile(0.25),
        "mediana": by_class.duration_s.median(),
        "q3": by_class.duration_s.quantile(0.75),
        "lo": by_class.low_freq_hz.quantile(0.25),
        "lo_med": by_class.low_freq_hz.median(),
        "hi_med": by_class.high_freq_hz.median(),
        "hi": by_class.high_freq_hz.quantile(0.75),
        "ancho": by_class.bandwidth_hz.median(),
        "n": by_class.size(),
    }
).sort_values("mediana", ascending=False)
order = geometry.index.tolist()
ys = np.arange(len(order))

fig, axes = plt.subplots(1, 2, figsize=(12.6, 0.28 * len(order) + 1.4), sharey=True, gridspec_kw={"wspace": 0.08})
ax = axes[0]
ax.set_xscale("log")
for y, (_, r) in zip(ys, geometry.iterrows()):
    ax.plot([r.q1, r.q3], [y, y], color=BLUE_FADED, linewidth=5, solid_capstyle="round", zorder=2)
    ax.scatter([r.mediana], [y], s=22, color=BLUE, zorder=3)
ax.axvline(P.clip_len_s, color=SERIES[1], linewidth=1.1, linestyle=(0, (4, 3)))
ax.text(P.clip_len_s * 1.08, -0.6, f"ventana {P.clip_len_s:g} s", color=SERIES[1], fontsize=8, va="center")
ax.set_ylim(len(order) - 0.4, -1.0)
ax.set_yticks(ys, order, fontsize=7.6)
ax.set_xlabel("duración (s, escala log): Q1 – mediana – Q3")
style_value_axis(ax, "x")

ax = axes[1]
for y, (_, r) in zip(ys, geometry.iterrows()):
    ax.plot([r.lo / 1000, r.hi / 1000], [y, y], color=GRID, linewidth=5, solid_capstyle="round", zorder=1)
    ax.plot([r.lo_med / 1000, r.hi_med / 1000], [y, y], color=SERIES[2], linewidth=2.2, zorder=3)
ax.set_xlim(0, P.f_max / 1000)
ax.set_xlabel("banda de frecuencia (kHz): mediana de f_min – f_max, y rango intercuartil")
style_value_axis(ax, "x")

save(
    fig,
    "class_geometry",
    data={
        "clase": order,
        "duracion_s": {"q1": geometry.q1, "mediana": geometry.mediana, "q3": geometry.q3},
        "banda_hz": {"pair": order, "lo": geometry.lo, "hi": geometry.hi, "lo_med": geometry.lo_med, "hi_med": geometry.hi_med},
    },
)
(DATA_DIR / "class_geometry_table.json").write_text(
    json.dumps(
        {
            "clase": order,
            "anotaciones": geometry.n,
            "duracion_mediana_ms": geometry.mediana * 1000,
            "banda_mediana_hz": [[lo, hi] for lo, hi in zip(geometry.lo_med, geometry.hi_med)],
            "ancho_de_banda_mediano_hz": geometry.ancho,
        },
        indent=1,
        ensure_ascii=False,
        default=_jsonable,
    )
)
plt.show()

In [ ]:
# Duración frente a ancho de banda, una faceta por clase: dónde una etiqueta esconde dos formas.
n_cols = 5
n_rows = int(np.ceil(len(CLASSES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13.0, 2.3 * n_rows), sharex=True, sharey=True)
for ax, pair in zip(axes.flat, CLASSES):
    rows = experiment[experiment.pair == pair]
    ax.scatter(rows.duration_s * 1000, rows.bandwidth_hz / 1000, s=5, color=BLUE, alpha=0.35, linewidths=0)
    ax.set_xscale("log")
    ax.set_title(f"{pair}  (n = {len(rows):,})", fontsize=8.5, loc="left", pad=4)
    ax.grid(True, color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
for ax in axes.flat[len(CLASSES):]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("duración (ms)")
for ax in axes[:, 0]:
    ax.set_ylabel("ancho de banda (kHz)")
fig.subplots_adjust(hspace=0.45, wspace=0.12)

save(fig, "class_geometry_facets")
plt.show()

In [ ]:
NESTED_PAIRS = [("lw/cc", "lw/cs"), ("sm/fc", "sm/fs"), ("sb/pcs", "sb/ppc")]

nesting_rows = []
for phrase_pair, syllable_pair in NESTED_PAIRS:
    phrases = annotations[annotations.pair == phrase_pair]
    syllables = annotations[annotations.pair == syllable_pair]
    if phrases.empty or syllables.empty:
        continue
    inside = 0
    for audio_path, group in syllables.groupby("audio_path"):
        window = phrases[phrases.audio_path == audio_path]
        if window.empty:
            continue
        mid = ((group.begin_time_s + group.end_time_s) / 2).to_numpy()[:, None]
        starts = window.begin_time_s.to_numpy()[None, :]
        ends = window.end_time_s.to_numpy()[None, :]
        inside += int(((mid >= starts) & (mid <= ends)).any(axis=1).sum())
    with_syllable = 0
    for audio_path, group in phrases.groupby("audio_path"):
        window = syllables[syllables.audio_path == audio_path]
        if window.empty:
            continue
        mid = ((window.begin_time_s + window.end_time_s) / 2).to_numpy()[None, :]
        starts = group.begin_time_s.to_numpy()[:, None]
        ends = group.end_time_s.to_numpy()[:, None]
        with_syllable += int(((mid >= starts) & (mid <= ends)).any(axis=1).sum())
    nesting_rows.append(
        {
            "frase": phrase_pair,
            "sílaba": syllable_pair,
            "frases": len(phrases),
            "frases con >=1 sílaba dentro": with_syllable / len(phrases),
            "sílabas": len(syllables),
            "sílabas dentro de una frase": inside / len(syllables),
        }
    )
nesting = pd.DataFrame(nesting_rows)

phrase_pair, syllable_pair = NESTED_PAIRS[0]
phrases = annotations[annotations.pair == phrase_pair]
best = None
for r in phrases.itertuples():
    contained = annotations[
        (annotations.audio_path == r.audio_path)
        & (annotations.pair == syllable_pair)
        & (annotations.begin_time_s >= r.begin_time_s - 0.05)
        & (annotations.end_time_s <= r.end_time_s + 0.05)
    ]
    if best is None or len(contained) > len(best[1]):
        best = (r, contained)
phrase, syllables = best

pad = max(phrase.duration_s * 0.18, 0.25)
start_s = max(phrase.begin_time_s - pad, 0.0)
duration_s = phrase.duration_s + 2 * pad
waveform = load_segment(phrase.audio_path, start_s, duration_s)

fig, ax = plt.subplots(figsize=(10.4, 4.8))
draw_spectrogram(ax, log_mel(waveform), duration_s)
draw_box(
    ax,
    phrase.begin_time_s - start_s,
    phrase.end_time_s - start_s,
    phrase.low_freq_hz,
    phrase.high_freq_hz,
    color=SERIES[1],
    label=f"{phrase_pair.upper()}  (frase, {phrase.duration_s:.1f} s)",
    lw=2.0,
    dashed=True,
)
for r in syllables.itertuples():
    draw_box(
        ax,
        r.begin_time_s - start_s,
        r.end_time_s - start_s,
        r.low_freq_hz,
        r.high_freq_hz,
        color=GOOD,
        lw=1.4,
    )
ax.text(
    (phrase.begin_time_s - start_s + phrase.end_time_s - start_s) / 2,
    0.06,
    f"{len(syllables)} sílabas {syllable_pair.upper()} dentro de una sola frase",
    ha="center",
    va="center",
    fontsize=8.5,
    color=GOOD,
    fontweight="bold",
    zorder=6,
    bbox=dict(boxstyle="round,pad=0.35", facecolor=SURFACE, edgecolor="none", alpha=0.92),
)
phrase_share = nesting.set_index("frase").loc[phrase_pair, "frases con >=1 sílaba dentro"]
save(fig, "nesting_phrase")
plt.show()

In [ ]:
N_COLS = 5
n_rows = int(np.ceil(len(CLASSES) / N_COLS))
order = experiment.groupby("pair").low_freq_hz.median().sort_values().index.tolist()

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(3.0 * N_COLS, 2.35 * n_rows))
axes_flat = axes.ravel()
for ax, pair in zip(axes_flat, order):
    row = pick_annotation(pair, df=experiment)
    span = max(row.duration_s * 2.4, 0.35)
    start_s = max(row.begin_time_s - (span - row.duration_s) / 2, 0.0)
    try:
        waveform = load_segment(row.audio_path, start_s, span)
        draw_spectrogram(ax, log_mel(waveform), span, n_ticks=4)
        draw_box(
            ax,
            row.begin_time_s - start_s,
            row.end_time_s - start_s,
            row.low_freq_hz,
            row.high_freq_hz,
            color=GOOD,
            lw=1.3,
        )
    except Exception as exc:  # noqa: BLE001
        ax.text(0.5, 0.5, f"sin audio\n{exc}", ha="center", va="center", fontsize=7, color=CRITICAL)
    ax.set_title(f"{pair.upper()}  ·  {CALL_NAME.get(pair, '')}", fontsize=8.5, pad=14, loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(labelsize=6.5)
    ax.text(
        0,
        1.015,
        f"n={int((experiment.pair == pair).sum()):,} anotaciones · mediana {row.duration_s * 1000:.0f} ms",
        transform=ax.transAxes,
        fontsize=6.8,
        color=INK_2,
        va="bottom",
    )

for ax in axes_flat[len(order) :]:
    ax.axis("off")

fig.tight_layout(rect=(0, 0, 1, 0.945))
save(fig, "call_gallery")
plt.show()

In [ ]:
BUILD_MANIFEST = True  # False para saltar el manifest (tarda un par de minutos)
from prepare_data import EMPTY_RATIO, SPLIT_RATIOS  # noqa: E402

manifest = splits = None
if BUILD_MANIFEST:
    from data.manifest import build_manifest, split_manifest  # noqa: E402
    from data.species import LabelSet  # noqa: E402

    labels = LabelSet(CLASSES)
    # Como `prepare_data.py`, salvo las ventanas de fondo de aves, que no son de una clase.
    manifest = build_manifest(experiment, labels, empty_ratio=EMPTY_RATIO, seed=42)
    splits = dict(
        zip(
            ("train", "val", "test"),
            split_manifest(manifest, seed=42, n_classes=len(labels), ratios=SPLIT_RATIOS),
        )
    )
    n_boxes = sum(len(w.boxes) for w in manifest)

In [ ]:
n_loaded = len(annotations)
n_pairs = annotations.pair.nunique()
excluded_mask = annotations.pair.isin(EXCLUDED_LABELS)
n_no_phrase = int((~excluded_mask).sum())
n_experiment = len(experiment)
n_review = int((~pd.Series(list(zip(annotations.species, annotations.call_type))).isin(VALID_PAIRS)).sum())

STEPS = [
    (
        "data/cleaned/",
        f"{len(recordings):,} grabaciones\n{recordings.has_annotation.sum():,} con .txt",
        "material recibido",
    ),
    (
        "Carga y consolidación",
        f"{n_loaded:,} anotaciones\n{n_pairs} pares distintos",
        "`load_annotations`; el .txt sin .wav se descarta",
    ),
    (
        "Normalización de etiquetas",
        f"{n_review:,} fuera del\nvocabulario",
        "sinónimos y `MANUAL_FIXES`",
    ),
    (
        "Saneamiento geométrico",
        f"recorte a Nyquist\n({P.f_max / 1000:.1f} kHz)",
        "fuera las cajas degeneradas",
    ),
    (
        "Vocabulario cerrado",
        f"{n_no_phrase:,} sin las 3\nclases de frase",
        "y los trinos de lw fusionados",
    ),
    (
        "Selección del subconjunto",
        f"{n_experiment:,} anotaciones\n{len(CLASSES)} clases",
        f"pares con >= {MIN_PAIR_COUNT}",
    ),
]
WINDOW_STEPS = [
    (
        "Ventaneo",
        f"{len(manifest):,} ventanas" if manifest else "ventanas de 3 s",
        f"clip {P.clip_len_s:g} s, salto {P.clip_hop_s:g} s",
    ),
    ("Partición por grabación", "60 / 22,5 / 17,5 %", "ninguna ventana cruza"),
    ("Caché de tensores", "train / val / test .pt", "mel de potencia y cajas cxcywh"),
]

fig, ax = canvas(12.6, 8.4)
TOP = 100 * 8.4 / 12.6
ROWS = [STEPS[:3], STEPS[3:], WINDOW_STEPS]
row_y = [TOP - 13, TOP - 36, TOP - 59]
placed = []
for r, (row, cy) in enumerate(zip(ROWS, row_y)):
    edge = SERIES[2] if r == 2 else BLUE
    placed.append(
        flow_row(
            ax,
            [{"title": name, "body": count, "foot": note} for name, count, note in row],
            cy,
            margin=6.5,
            gap=3.4,
            wrap=24,
            fill=PANEL if r == 2 else SURFACE,
            edge=edge,
            arrow_color=edge,
            lw=1.3,
            fs=9.0,
            body_fs=7.8,
            foot_fs=6.9,
        )
    )
for r in range(2):
    cx_last, cy_last, w_last, h_last = placed[r][-1]
    cx_first, cy_next, w_first, h_next = placed[r + 1][0]
    mid = (cy_last - h_last / 2 + cy_next + h_next / 2) / 2
    elbow(
        ax,
        [
            (cx_last, cy_last - h_last / 2),
            (cx_last, mid),
            (cx_first, mid),
            (cx_first, cy_next + h_next / 2),
        ],
        color=SERIES[2] if r == 1 else BLUE,
    )

chip(ax, 97.4, row_y[0], "RE1.1", BLUE, fs=7.5)
chip(ax, 97.4, row_y[2], "RE1.2", SERIES[2], fs=7.5)
save(fig, "curation_pipeline")
plt.show()

In [ ]:
fig, ax = canvas(12.0, 4.8)
TOP = 100 * 4.8 / 12.0
BLOCK_Y = TOP - 19

for x0, title, ok in [(3, "Partición por ventana", False), (54, "Partición por grabación", True)]:
    caption_at(ax, x0, TOP - 4, title, fs=10, color=INK, ha="left", weight="bold")
    width = 40
    ax.add_patch(
        FancyBboxPatch(
            (x0, BLOCK_Y),
            width,
            6,
            boxstyle="round,pad=0,rounding_size=0.8",
            facecolor=PANEL,
            edgecolor=BASELINE,
            linewidth=1.1,
        )
    )
    caption_at(ax, x0 + width - 4, BLOCK_Y - 3.0, "una grabación", fs=7.5, ha="right")
    assignment = ["train"] * 4 if ok else ["train", "test", "train", "val"]
    tint = {"train": BLUE, "val": SERIES[1], "test": SERIES[2]}
    for i in range(4):
        start = x0 + 1.5 + i * 8.5
        ax.add_patch(
            FancyBboxPatch(
                (start, BLOCK_Y + 1 + (i % 2) * 2),
                14,
                2.6,
                boxstyle="round,pad=0,rounding_size=0.5",
                facecolor=tint[assignment[i]],
                alpha=0.6,
                edgecolor=tint[assignment[i]],
                linewidth=1.0,
            )
        )
        caption_at(
            ax,
            start + 7,
            BLOCK_Y + 2.3 + (i % 2) * 2,
            assignment[i],
            fs=6.4,
            color=SURFACE if assignment[i] == "train" else INK,
        )
    if not ok:
        ax.add_patch(
            FancyBboxPatch(
                (x0 + 9.5, BLOCK_Y - 1.5),
                6,
                9,
                boxstyle="round,pad=0,rounding_size=0.5",
                facecolor="none",
                edgecolor=CRITICAL,
                linewidth=1.4,
                linestyle=(0, (3, 2)),
            )
        )
        caption_at(
            ax,
            x0 + 12.5,
            BLOCK_Y - 6.5,
            "audio compartido\nentre particiones",
            fs=7,
            color=CRITICAL,
            va="top",
        )

save(fig, "split_by_recording")
plt.show()

In [ ]:
# Cajas por clase y partición, con el total de anotaciones de la clase como referencia: lo
# que el ventaneo deja afuera (llamadas más largas que la ventana) se ve en la diferencia.
if splits is None:
    print("sin manifest: activá BUILD_MANIFEST")
else:
    per_split = {}
    for name, windows in splits.items():
        ids = np.concatenate([w.labels for w in windows]) if windows else np.zeros(0, dtype=int)
        per_split[name] = np.bincount(ids, minlength=len(labels))
    table = pd.DataFrame(per_split, index=labels.names)
    table["annotations"] = experiment.pair.value_counts().reindex(labels.names).fillna(0).astype(int)
    table = table.sort_values("train", ascending=False)
    ys = np.arange(len(table))

    fig, ax = plt.subplots(figsize=(9.6, 0.28 * len(table) + 1.4))
    left = np.zeros(len(table))
    for name, color in zip(("train", "val", "test"), (BLUE, SERIES[2], SERIES[1])):
        barh(ax, ys, table[name].values, color, height=0.62, left=left, label=name)
        left = left + table[name].values
    ax.scatter(table["annotations"], ys, s=16, color=INK, zorder=4, label="anotaciones")
    for y, total in zip(ys, left):
        ax.text(total + left.max() * 0.012, y, f"{int(total):,}", va="center", fontsize=6.8, color=INK_2)
    ax.set_xlim(0, left.max() * 1.15)
    ax.set_ylim(len(table) - 0.4, -0.8)
    ax.set_yticks(ys, table.index, fontsize=7.6)
    ax.set_xlabel("cajas en el manifest")
    ax.legend(loc="lower right", ncol=4)
    style_value_axis(ax, "x")

    save(
        fig,
        "boxes_per_split",
        data={"clase": table.index, **{k: table[k] for k in ("train", "val", "test", "annotations")}},
    )
    plt.show()

In [ ]:
fig, ax = canvas(12.8, 6.8)
TOP = 100 * 6.8 / 12.8
SPAN_S = 12.0
X0, TIMELINE_W = 3.0, 62.0
SCALE = TIMELINE_W / SPAN_S
NOTE_X = X0 + TIMELINE_W + 3.0
AXIS_Y = TOP - 16


def t2x(t):
    return X0 + t * SCALE


starts = np.arange(0, SPAN_S - P.clip_len_s + 0.01, P.clip_hop_s)
for i, start in enumerate(starts):
    y = AXIS_Y + 3.0 + (i % 2) * 4.0
    ax.add_patch(
        FancyBboxPatch(
            (t2x(start), y),
            P.clip_len_s * SCALE,
            3.2,
            boxstyle="round,pad=0,rounding_size=0.7",
            facecolor=BLUE,
            alpha=0.16 + 0.06 * (i % 2),
            edgecolor=BLUE,
            linewidth=1.0,
        )
    )
    caption_at(ax, t2x(start + P.clip_len_s / 2), y + 1.6, f"w{i}", fs=6.8, color=BLUE)
caption_at(
    ax,
    NOTE_X,
    AXIS_Y + 5.0,
    f"ventanas de {P.clip_len_s:g} s\ncon salto de {P.clip_hop_s:g} s",
    fs=7.8,
    color=BLUE,
    ha="left",
    weight="bold",
)

ax.plot([X0, X0 + SPAN_S * SCALE], [AXIS_Y, AXIS_Y], color=BASELINE, linewidth=1.2)
for t in range(0, int(SPAN_S) + 1):
    ax.plot([t2x(t), t2x(t)], [AXIS_Y - 0.8, AXIS_Y], color=BASELINE, linewidth=0.9)
    if t % 3 == 0:
        caption_at(ax, t2x(t), AXIS_Y - 3.0, f"{t} s", fs=7.5)

EVENTS = [
    (0.6, 1.5, "Dentro de una ventana", GOOD, "entra completo en w0"),
    (
        4.2,
        5.6,
        "A caballo de dos",
        GOOD,
        "entra en w2 y en w3: la misma anotación produce dos cajas",
    ),
    (
        7.3,
        8.0,
        "Cerca del borde",
        WARNING,
        f"sólo entra donde el solape alcanza min_overlap = {P.min_overlap:g}",
    ),
    (
        2.0,
        10.5,
        "Más largo que dos clips",
        CRITICAL,
        f"nunca alcanza {P.min_overlap:g} de su duración en ninguna ventana: inalcanzable",
    ),
]
pitch = (AXIS_Y - 7.0) / len(EVENTS)
for i, (t0, t1, name, color, note) in enumerate(EVENTS):
    y = AXIS_Y - 7.0 - i * pitch
    ax.add_patch(
        FancyBboxPatch(
            (t2x(t0), y),
            (t1 - t0) * SCALE,
            3.2,
            boxstyle="round,pad=0,rounding_size=0.7",
            facecolor=color,
            alpha=0.22,
            edgecolor=color,
            linewidth=1.4,
        )
    )
    caption_at(ax, t2x(t0) + 0.8, y + 1.6, f"{t1 - t0:.1f} s", fs=7, color=INK_2, ha="left")
    caption_at(
        ax, NOTE_X, y + 3.4, f"{name}\n{note}", fs=7.4, color=color, ha="left", wrap=30, va="top"
    )

save(fig, "windowing")
plt.show()

In [ ]:
# Cuántas cajas caen en una misma ventana de 3 s (sólo ventanas con al menos una).
if manifest is None:
    print("sin manifest: activá BUILD_MANIFEST")
else:
    per_window = np.array([len(w.boxes) for w in manifest if len(w.boxes)])
    values, counts = np.unique(per_window, return_counts=True)
    fig, ax = plt.subplots(figsize=(8.4, 3.8))
    barv(ax, values, counts, BLUE, width=0.7)
    for v, c in zip(values, counts):
        ax.text(v, c * 1.25, f"{c:,}", ha="center", va="bottom", fontsize=7.2, color=INK_2)
    ax.set_yscale("log")
    ax.set_xlim(values.min() - 0.7, values.max() + 0.7)
    ax.set_ylim(0.7, counts.max() * 3)
    ax.set_xticks(values)
    ax.set_xlabel("cajas en la ventana")
    ax.set_ylabel("ventanas (escala log)")
    style_value_axis(ax, "y")
    save(fig, "boxes_per_window", data={"cajas_en_la_ventana": values, "ventanas": counts})
    plt.show()

In [ ]:
# Una ventana de entrenamiento tal como la ve el modelo: mel de 3 s y sus cajas cxcywh.
if manifest is None:
    print("sin manifest: activá BUILD_MANIFEST")
else:
    busy = sorted((w for w in manifest if 3 <= len(w.boxes) <= 5), key=lambda w: -len(set(w.labels.tolist())))
    window = busy[0]
    waveform = load_segment(window.audio_path, window.clip_start_s, P.clip_len_s)
    fig, ax = plt.subplots(figsize=(9.0, 4.2))
    draw_spectrogram(ax, log_mel(waveform), P.clip_len_s)
    for (cx, cy, w, h), class_id in zip(window.boxes, window.labels):
        t0, t1 = (cx - w / 2) * P.clip_len_s, (cx + w / 2) * P.clip_len_s
        low, high = y_to_hz(np.array([cy - h / 2, cy + h / 2]), P)
        draw_box(ax, t0, t1, low, high, color=GOOD, label=labels.name(class_id).upper(), lw=1.5)
    save(fig, "window_example")
    plt.show()